<a href="https://colab.research.google.com/github/jsemprini/Iowa-Water-Nitrate-Births-8488update/blob/main/Copy_of_4A_prepare_analysis_final_T1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# File 4A (Final V3) — Freeze the final T1 analysis dataset

**Purpose.** Create the final compact analysis file after T1 linkage. This notebook does not define study outcomes or estimate statistical models.

It creates only the data file, performs final QA, and saves a dataset containing analytic variables:

`county_fips birth_year gest_age_weeks conception_quarter birthweight_g infant_male maternal_age maternal_race_broad married prenatal_by5 t1_mean_complete t1_mean_observed parity2 parity3plus meduc_hs meduc_gt_hs`


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from google.colab import drive

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 250)
pd.set_option('display.width', 260)
drive.mount('/content/drive', force_remount=False)


Mounted at /content/drive


In [ ]:
# ============================================================
# 0. PATHS + LOAD
# ============================================================
INPUT_PARQUET = '/content/drive/MyDrive/plos-update-v3/3-linked/birth_water_linked_T1_v3.parquet'
INPUT_CSV = '/content/drive/MyDrive/plos-update-v3/3-linked/birth_water_linked_T1_v3.csv'
OUTPUT_ROOT = '/content/drive/MyDrive/plos-update-v3/4-analysis'
DATA_DIR = os.path.join(OUTPUT_ROOT, 'data')
TABLE_DIR = os.path.join(OUTPUT_ROOT, 'tables')
STATA_DIR = os.path.join(OUTPUT_ROOT, 'stata')
for d in [OUTPUT_ROOT, DATA_DIR, TABLE_DIR, STATA_DIR]:
    os.makedirs(d, exist_ok=True)

if os.path.exists(INPUT_PARQUET):
    df = pd.read_parquet(INPUT_PARQUET)
elif os.path.exists(INPUT_CSV):
    df = pd.read_csv(INPUT_CSV, low_memory=False)
else:
    raise FileNotFoundError('Neither File 3 Parquet nor CSV input was found.')

print('Rows loaded:', f'{len(df):,}')


Rows loaded: 164,977


In [ ]:
# ============================================================
# 1. VALIDATE INPUT + CREATE ONLY REQUESTED DERIVED COVARIATES
# ============================================================
required = [
    'county_fips', 'birth_year', 'gest_age_weeks', 'conception_quarter',
    'birthweight_g', 'infant_male', 'maternal_age', 'maternal_race_broad',
    'married', 'prenatal_by5', 't1_mean_complete', 't1_mean_observed',
    'live_birth_order', 'maternal_education_years'
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError('Required File 3 variables missing: ' + ', '.join(missing))

numeric_cols = [c for c in required if c != 'maternal_race_broad']
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')


def indicator(condition, valid):
    condition = pd.Series(condition, index=df.index).astype('boolean')
    valid = pd.Series(valid, index=df.index).fillna(False).astype(bool)
    out = pd.Series(np.nan, index=df.index, dtype=float)
    out.loc[valid] = condition.loc[valid].fillna(False).astype(float)
    return out

# Parity: first live birth is the omitted/reference category.
parity_valid = df['live_birth_order'].notna()
df['parity2'] = indicator(df['live_birth_order'].eq(2), parity_valid)
df['parity3plus'] = indicator(df['live_birth_order'] >= 3, parity_valid)

# Maternal education: <12 years is the omitted/reference category.
edu_valid = df['maternal_education_years'].notna()
df['meduc_hs'] = indicator(df['maternal_education_years'].eq(12), edu_valid)
df['meduc_gt_hs'] = indicator(df['maternal_education_years'] > 12, edu_valid)

print('Derived parity and maternal-education indicators.')


Derived parity and maternal-education indicators.


In [ ]:
# ============================================================
# 2. FINAL QUALITY CONTROL / DIAGNOSTICS
# ============================================================
# These diagnostics are saved separately; they do not add columns to the final file.
final_qa = pd.DataFrame({
    'metric': [
        'n_rows',
        'n_counties',
        'n_birth_years',
        'pct_missing_t1_mean_complete',
        'pct_missing_t1_mean_observed',
        'mean_t1_mean_complete',
        'mean_t1_mean_observed_among_available',
        'correlation_complete_vs_observed_among_available',
        'pct_missing_gest_age_weeks',
        'pct_missing_birthweight_g',
        'pct_missing_maternal_age',
        'pct_missing_maternal_race_broad',
        'pct_missing_prenatal_by5',
        'pct_missing_parity_indicators',
        'pct_missing_maternal_education_indicators',
    ],
    'value': [
        len(df),
        df['county_fips'].nunique(dropna=True),
        df['birth_year'].nunique(dropna=True),
        100 * df['t1_mean_complete'].isna().mean(),
        100 * df['t1_mean_observed'].isna().mean(),
        df['t1_mean_complete'].mean(),
        df['t1_mean_observed'].mean(),
        df[['t1_mean_complete', 't1_mean_observed']].corr().iloc[0, 1],
        100 * df['gest_age_weeks'].isna().mean(),
        100 * df['birthweight_g'].isna().mean(),
        100 * df['maternal_age'].isna().mean(),
        100 * df['maternal_race_broad'].isna().mean(),
        100 * df['prenatal_by5'].isna().mean(),
        100 * (df['parity2'].isna() | df['parity3plus'].isna()).mean(),
        100 * (df['meduc_hs'].isna() | df['meduc_gt_hs'].isna()).mean(),
    ]
})
display(final_qa)

# Compact summaries by birth year and maternal race for QA only.
qa_by_birth_year = (
    df.groupby('birth_year', dropna=False, as_index=False)
    .agg(
        n=('county_fips', 'size'),
        mean_t1_complete=('t1_mean_complete', 'mean'),
        mean_t1_observed=('t1_mean_observed', 'mean'),
        mean_gest_age=('gest_age_weeks', 'mean'),
        mean_birthweight=('birthweight_g', 'mean'),
    )
)
display(qa_by_birth_year)

qa_by_race = (
    df.groupby('maternal_race_broad', dropna=False, as_index=False)
    .agg(
        n=('county_fips', 'size'),
        mean_t1_complete=('t1_mean_complete', 'mean'),
        mean_gest_age=('gest_age_weeks', 'mean'),
        mean_birthweight=('birthweight_g', 'mean'),
    )
)
display(qa_by_race)


,metric,value
0,n_rows,164977.000000
1,n_counties,99.000000
2,n_birth_years,6.000000
3,pct_missing_t1_mean_complete,0.000000
4,pct_missing_t1_mean_observed,11.550095
5,mean_t1_mean_complete,3.467369
6,mean_t1_mean_observed_among_available,3.564306
7,correlation_complete_vs_observed_among_available,0.982730
8,pct_missing_gest_age_weeks,0.000000
9,pct_missing_birthweight_g,0.027883


,birth_year,n,mean_t1_complete,mean_t1_observed,mean_gest_age,mean_birthweight
0,1983,7951,2.897623,2.975913,38.735505,3390.775009
1,1984,33576,3.626771,3.793751,39.057482,3432.845899
2,1985,32799,3.139667,3.196492,39.011128,3433.224004
3,1986,31325,3.283039,3.412555,39.007023,3435.041067
4,1987,30387,3.722609,3.872641,38.954290,3434.169185
5,1988,28939,3.741890,3.775483,39.039255,3439.314196


,maternal_race_broad,n,mean_t1_complete,mean_gest_age,mean_birthweight
0,American Indian,508,5.745028,38.742126,3444.039370
1,Asian/Pacific Islander,1771,3.840470,38.513834,3211.080181
2,Black,3752,3.535652,38.094616,3128.715733
3,Other nonwhite,5,4.293982,39.600000,3586.600000
4,White,158887,3.454260,39.028674,3442.313121
5,NaN,54,3.552416,38.796296,3379.351852


In [ ]:
# ============================================================
# 3. KEEP EXACTLY THE REQUESTED 16 VARIABLES
# ============================================================
FINAL_VARS = [
    'county_fips',
    'birth_year',
    'gest_age_weeks',
    'conception_quarter',
    'birthweight_g',
    'infant_male',
    'maternal_age',
    'maternal_race_broad',
    'married',
    'prenatal_by5',
    't1_mean_complete',
    't1_mean_observed',
    'parity2',
    'parity3plus',
    'meduc_hs',
    'meduc_gt_hs',
]

final_df = df[FINAL_VARS].copy()

# Hard safeguards: final analysis data must contain exactly and only these variables.
assert list(final_df.columns) == FINAL_VARS
assert final_df.shape[1] == 16
assert final_df['county_fips'].notna().all()
assert final_df['gest_age_weeks'].between(20, 41).all()
assert final_df['t1_mean_complete'].notna().all()
assert (final_df['t1_mean_complete'] >= 0).all()

# Use standard integer dtypes where the cohort guarantees nonmissing values.
final_df['county_fips'] = pd.to_numeric(final_df['county_fips'], errors='raise').astype(np.int64)
final_df['birth_year'] = pd.to_numeric(final_df['birth_year'], errors='raise').astype(np.int16)
final_df['conception_quarter'] = pd.to_numeric(final_df['conception_quarter'], errors='raise').astype(np.int8)

# Keep race as ordinary Python strings/None for portable Stata export.
final_df['maternal_race_broad'] = final_df['maternal_race_broad'].where(
    final_df['maternal_race_broad'].notna(), None
).astype(object)

print('Final columns:')
print(list(final_df.columns))
print('Final rows:', f'{len(final_df):,}')


Final columns:
['county_fips', 'birth_year', 'gest_age_weeks', 'conception_quarter', 'birthweight_g', 'infant_male', 'maternal_age', 'maternal_race_broad', 'married', 'prenatal_by5', 't1_mean_complete', 't1_mean_observed', 'parity2', 'parity3plus', 'meduc_hs', 'meduc_gt_hs']
Final rows: 164,977


In [ ]:
# ============================================================
# 4. SAVE FINAL ANALYSIS DATA + QA
# ============================================================
ANALYSIS_PARQUET = os.path.join(DATA_DIR, 'analysis_ready_T1_v3.parquet')
ANALYSIS_CSV = os.path.join(DATA_DIR, 'analysis_ready_T1_v3.csv')
ANALYSIS_STATA = os.path.join(STATA_DIR, 'analysis_ready_T1_v3.dta')

final_df.to_parquet(ANALYSIS_PARQUET, index=False)
final_df.to_csv(ANALYSIS_CSV, index=False)
try:
    final_df.to_stata(ANALYSIS_STATA, write_index=False, version=118)
except Exception as exc:
    print('Stata export skipped:', exc)

final_qa.to_csv(os.path.join(TABLE_DIR, 'final_data_qa_T1_v3.csv'), index=False)
qa_by_birth_year.to_csv(os.path.join(TABLE_DIR, 'final_data_qa_by_birth_year_T1_v3.csv'), index=False)
qa_by_race.to_csv(os.path.join(TABLE_DIR, 'final_data_qa_by_race_T1_v3.csv'), index=False)

print('File 4A Final V3 complete.')
print('Parquet:', ANALYSIS_PARQUET)
print('CSV:', ANALYSIS_CSV)
print('Stata:', ANALYSIS_STATA)


File 4A Final V3 complete.
Parquet: /content/drive/MyDrive/plos-update-v3/4-analysis/data/analysis_ready_T1_v3.parquet
CSV: /content/drive/MyDrive/plos-update-v3/4-analysis/data/analysis_ready_T1_v3.csv
Stata: /content/drive/MyDrive/plos-update-v3/4-analysis/stata/analysis_ready_T1_v3.dta
